# Pool Sessions Analysis - Organized

This notebook provides a well-organized pipeline for analyzing neural data across multiple sessions.
It includes attribution analysis, R² scoring, spectral analysis, and visualization functions.

## Overview
1. **Setup & Data Loading**: Import libraries and load session data
2. **Channel Alignment**: Align channels across sessions using layer 4 borders
3. **Attribution Analysis**: Process and analyze attribution scores
4. **R² Analysis**: Evaluate model performance metrics
5. **Spectral Analysis**: Analyze frequency-specific patterns
6. **Visualization**: Generate comprehensive plots and figures

## 1. Setup and Configuration

In [ ]:
# Import required libraries
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import pandas as pd
import sys
import os
from plots import *
import warnings
import seaborn as sns
from scipy import stats
from scipy.stats import zscore
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d

# Configure matplotlib for PDF output
plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'],
    'font.size': 10,
    'text.usetex': False
})

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
# Define analysis parameters and paths
class Config:
    """Configuration class for analysis parameters and paths."""
    
    # Analysis parameters
    CHANNELS_TO_KEEP = 16
    BANDS = [(0.5, 4), (4, 8), (8, 12), (12, 25), (25, 50), (50, 100), (100, 200), (200, 400)]
    
    # Data paths
    BASE_PATH = Path(r'Y:\buzsakilab\Buzsakilabspace\LabShare\NoamNitzan\Open_Access\Allen_2022\spikes2lfp')
    ATTR_PATH = BASE_PATH / 'avr_attrs'
    SPONT_ATTR_PATH = BASE_PATH / 'avr_attrs_spont'
    R2_SCORES_PATH = BASE_PATH / 'r2_all'
    SPECTRAL_R2_PATH = BASE_PATH / 'spectral_R2_analysis_all'
    REGIONAL_DOMINANCE_PATH = BASE_PATH / 'regional_dominance_scores'
    STIM_PSTH_PATH = BASE_PATH / 'stim_psth'
    FIGURES_PATH = BASE_PATH / 'figures'
    
    # Table paths
    UNITS_INFO_PATH = 'tables/units_info.csv'
    BORDERS_PATH = 'tables/layer_4_borders.csv'
    VISP_DEPTHS_PATH = 'tables/VISp_depths.csv'
    UNITS_ALL_PATH = BASE_PATH.parent / 'units.csv'

config = Config()
print(f"Analysis configured for {config.CHANNELS_TO_KEEP} channels and {len(config.BANDS)} frequency bands")

## 2. Data Loading Functions

In [ ]:
def load_and_prepare_dataframes():
    """Load and prepare all required dataframes with proper preprocessing.
    
    Returns:
        tuple: (df_units, borders_df, visp_depths_df, all_sessions)
    """
    print("Loading dataframes...")
    
    # Load main units dataframe
    df = pd.read_csv(config.UNITS_INFO_PATH)
    
    # Add connectivity features
    df['has_output'] = (
        (df['num_conn_e_to_e'] > 0) | (df['num_conn_e_to_i'] > 0) | 
        (df['num_conn_i_to_e'] > 0) | (df['num_conn_i_to_i'] > 0)
    ).astype(int)
    
    df['receives_input'] = (
        (df['num_conn_e_from_e'] > 0) | (df['num_conn_i_from_e'] > 0) | 
        (df['num_conn_e_from_i'] > 0) | (df['num_conn_i_from_i'] > 0)
    ).astype(int)
    
    # Load supplementary data
    units_all = pd.read_csv(config.UNITS_ALL_PATH)
    borders = pd.read_csv(config.BORDERS_PATH)
    visp_depths = pd.read_csv(config.VISP_DEPTHS_PATH)
    
    # Merge structure information
    df = df.merge(units_all[['unit_id', 'structure_acronym']], on='unit_id', how='left')
    
    # Standardize brain region names
    region_mapping = {
        ('LGd', 'LGv'): 'LG',
        ('MGv', 'MGd', 'MGm'): 'MG', 
        ('SCig', 'SCiw'): 'SC'
    }
    
    for old_regions, new_region in region_mapping.items():
        df.loc[df['structure_acronym'].isin(old_regions), 'structure_acronym'] = new_region
    
    all_sessions = visp_depths['session'].unique().tolist()
    
    print(f"Loaded {len(df)} units across {len(all_sessions)} sessions")
    print(f"Brain regions: {sorted(df['structure_acronym'].unique())}")
    
    return df, borders, visp_depths, all_sessions

## 3. Channel Alignment Functions

In [ ]:
def calculate_channel_borders(borders_df, visp_depths_df):
    """Calculate aligned channel borders for all sessions.
    
    Args:
        borders_df: DataFrame with border information
        visp_depths_df: DataFrame with channel depth information
        
    Returns:
        DataFrame: Updated borders with alignment indices
    """
    print("Calculating channel borders for alignment...")
    borders_updated = borders_df.copy()
    
    for i, row in borders_updated.iterrows():
        session = row['session']
        
        if session not in visp_depths_df['session'].values:
            continue
            
        ses_channels = visp_depths_df[visp_depths_df['session'] == session]
        
        # Find border indices
        upper_border_idx = np.where(ses_channels['depths'] == row['upper'])[0]
        if len(upper_border_idx) == 0:
            continue
        upper_border_idx = upper_border_idx[0]
        
        num_channels = len(ses_channels)
        
        # Calculate channel range: upper_border ± 5 and ± 10 channels
        most_superficial_idx = min(num_channels - 1, upper_border_idx + 5)
        deepest_idx = max(0, upper_border_idx - 10)
        
        # For trimming arrays
        first_channel_idx = deepest_idx
        last_channel_idx = most_superficial_idx
        
        # Calculate alignment start index
        start_idx = max(0, 10 - upper_border_idx) if upper_border_idx < 10 else 0
        
        # Update borders dataframe
        borders_updated.loc[i, 'upper_border_index'] = upper_border_idx
        borders_updated.loc[i, 'first_channel_index'] = first_channel_idx
        borders_updated.loc[i, 'last_channel_index'] = last_channel_idx
        borders_updated.loc[i, 'start_index'] = start_idx
    
    # Convert to proper integer types
    int_columns = ['upper_border_index', 'first_channel_index', 'last_channel_index', 'start_index']
    for col in int_columns:
        if col in borders_updated.columns:
            borders_updated[col] = borders_updated[col].astype('Int64')
    
    # Save updated borders
    borders_updated.to_csv('tables/layer_4_borders_aligned.csv', index=False)
    
    avg_channels = np.mean(borders_updated['upper_border_index'] - borders_updated['first_channel_index'])
    print(f"Average channels between borders: {int(np.ceil(avg_channels))}")
    
    return borders_updated

In [ ]:
def trim_session_data(data_array, session_id, borders_df, visp_depths_df, output_channels=None):
    """Trim data array according to calculated channel borders.
    
    Args:
        data_array: Input array to trim (channels x ...)
        session_id: Session identifier
        borders_df: DataFrame with border information
        visp_depths_df: DataFrame with channel depths
        output_channels: Number of output channels (default: config.CHANNELS_TO_KEEP)
        
    Returns:
        np.ndarray: Trimmed array aligned to standard channel grid
    """
    if output_channels is None:
        output_channels = config.CHANNELS_TO_KEEP
        
    # Get session border information
    session_borders = borders_df[borders_df['session'] == session_id]
    if session_borders.empty:
        return None
        
    row = session_borders.iloc[0]
    
    # Handle potential size mismatch between data and expected channels
    expected_channels = len(visp_depths_df[visp_depths_df['session'] == session_id])
    if data_array.shape[0] > expected_channels:
        diff = data_array.shape[0] - expected_channels
        data_array = data_array[:-diff]
    
    # Extract border indices
    start_index = int(row['start_index'])
    first_channel_index = int(row['first_channel_index'])
    last_channel_index = int(row['last_channel_index'])
    
    # Calculate channels to copy with bounds checking
    available_channels = data_array.shape[0]
    channels_to_copy = last_channel_index - first_channel_index + 1
    
    if first_channel_index + channels_to_copy > available_channels:
        channels_to_copy = available_channels - first_channel_index
    if start_index + channels_to_copy > output_channels:
        channels_to_copy = output_channels - start_index
    
    # Initialize output array with NaN
    output_shape = (output_channels,) + data_array.shape[1:]
    trimmed_array = np.full(output_shape, np.nan)
    
    # Copy data if valid range
    if channels_to_copy > 0:
        end_source = first_channel_index + channels_to_copy
        end_target = start_index + channels_to_copy
        trimmed_array[start_index:end_target] = data_array[first_channel_index:end_source]
    
    return trimmed_array

## 4. Attribution Data Processing

In [ ]:
def load_attribution_data(borders_df, visp_depths_df):
    """Load and align attribution data across all sessions.
    
    Args:
        borders_df: DataFrame with border information
        visp_depths_df: DataFrame with channel depths
        
    Returns:
        tuple: (attribution_dict, spontaneous_attribution_dict)
    """
    print("Loading attribution data...")
    
    attribution_data = {}
    spont_attribution_data = {}
    
    session_count = 0
    
    for _, row in borders_df.iterrows():
        session = row['session']
        
        if session not in visp_depths_df['session'].values:
            continue
            
        # Define file paths
        attr_file = config.ATTR_PATH / f'{session}_attribution_scores_ipi_mean.npy'
        spont_attr_file = config.SPONT_ATTR_PATH / f'{session}_attribution_scores_spontaneous_ipi_mean.npy'
        
        if not attr_file.exists() or not spont_attr_file.exists():
            continue
            
        # Load attribution arrays
        attr_array = np.load(attr_file)
        spont_attr_array = np.load(spont_attr_file)
        
        # Trim and align arrays
        trimmed_attr = trim_session_data(attr_array, session, borders_df, visp_depths_df)
        trimmed_spont = trim_session_data(spont_attr_array, session, borders_df, visp_depths_df)
        
        if trimmed_attr is not None and trimmed_spont is not None:
            attribution_data[session] = trimmed_attr
            spont_attribution_data[session] = trimmed_spont
            session_count += 1
    
    print(f"Loaded attribution data for {session_count} sessions")
    print(f"Data shape per session: {list(attribution_data.values())[0].shape if attribution_data else 'N/A'}")
    
    return attribution_data, spont_attribution_data

## 5. Main Analysis Pipeline

In [ ]:
# Load and prepare all data
df_units, borders_df, visp_depths_df, all_sessions = load_and_prepare_dataframes()

# Calculate channel alignments
borders_aligned = calculate_channel_borders(borders_df, visp_depths_df)

# Load attribution data
attribution_data, spont_attribution_data = load_attribution_data(borders_aligned, visp_depths_df)

print("\n=== Data Loading Complete ===")
print(f"Units: {len(df_units)}")
print(f"Sessions: {len(attribution_data)}")
print(f"Channels per session: {config.CHANNELS_TO_KEEP}")
print(f"Frequency bands: {len(config.BANDS) + 1} (including broadband)")

## 6. Basic Visualization

In [ ]:
def plot_session_overview(attribution_data):
    """Create overview plot of attribution data across sessions.
    
    Args:
        attribution_data: Dictionary of attribution arrays by session
    """
    sessions = list(attribution_data.keys())
    n_sessions = len(sessions)
    
    # Calculate mean attribution per session (broadband)
    session_means = []
    session_stds = []
    
    for session in sessions:
        attr_data = attribution_data[session][:, 0, :]  # broadband, all units
        session_means.append(np.nanmean(np.abs(attr_data)))
        session_stds.append(np.nanstd(np.abs(attr_data)))
    
    # Create summary plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Session means
    ax1.bar(range(n_sessions), session_means, yerr=session_stds, 
            alpha=0.7, color='steelblue', capsize=3)
    ax1.set_xlabel('Session Index')
    ax1.set_ylabel('Mean Attribution Score')
    ax1.set_title(f'Attribution Scores Across {n_sessions} Sessions (Broadband)')
    ax1.grid(axis='y', alpha=0.3)
    
    # Distribution of session means
    ax2.hist(session_means, bins=10, alpha=0.7, color='coral', edgecolor='black')
    ax2.axvline(np.mean(session_means), color='red', linestyle='--', 
                label=f'Mean: {np.mean(session_means):.2e}')
    ax2.set_xlabel('Mean Attribution Score')
    ax2.set_ylabel('Number of Sessions')
    ax2.set_title('Distribution of Session-level Means')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return session_means

# Generate overview plot
if attribution_data:
    session_overview = plot_session_overview(attribution_data)
else:
    print("No attribution data available for plotting")

---
## Next Steps

This notebook provides the core structure and data loading pipeline. To continue the analysis, you can:

1. **Add R² Analysis**: Load and process R² scores with visualization
2. **Implement Attribution Weighting**: Correct for R² correlations
3. **Add Spectral Analysis**: Process frequency-specific data
4. **Build Area-wise Analysis**: Group by brain regions and calculate statistics
5. **Add Advanced Plotting**: Create publication-quality figures

Run this notebook first to load the core data, then we can add additional analysis modules.

## 7. R² Analysis Functions

In [ ]:
def load_r2_scores(borders_df, visp_depths_df):
    """Load and align R² scores across sessions.
    
    Args:
        borders_df: DataFrame with border information
        visp_depths_df: DataFrame with channel depths
        
    Returns:
        tuple: (r2_active, r2_spontaneous, r2_passive)
    """
    print("Loading R² scores...")
    
    r2_active = {}
    r2_spontaneous = {}
    r2_passive = {}
    
    for _, row in borders_df.iterrows():
        session = row['session']
        
        if session not in visp_depths_df['session'].values:
            continue
            
        # Define file paths
        r2_file = config.R2_SCORES_PATH / f'{session}_r2_scores.npy'
        r2_spont_file = config.R2_SCORES_PATH / f'{session}_r2_scores_spont.npy'
        r2_passive_file = config.R2_SCORES_PATH / f'{session}_r2_scores_passive.npy'
        
        if not all([f.exists() for f in [r2_file, r2_spont_file, r2_passive_file]]):
            continue
            
        # Load R² arrays
        r2_array = np.load(r2_file)
        r2_spont_array = np.load(r2_spont_file)
        r2_pass_array = np.load(r2_passive_file)
        
        # Trim and align arrays
        trimmed_r2 = trim_session_data(r2_array, session, borders_df, visp_depths_df)
        trimmed_r2_spont = trim_session_data(r2_spont_array, session, borders_df, visp_depths_df)
        trimmed_r2_pass = trim_session_data(r2_pass_array, session, borders_df, visp_depths_df)
        
        if all([arr is not None for arr in [trimmed_r2, trimmed_r2_spont, trimmed_r2_pass]]):
            r2_active[session] = trimmed_r2
            r2_spontaneous[session] = trimmed_r2_spont
            r2_passive[session] = trimmed_r2_pass
    
    print(f"Loaded R² scores for {len(r2_active)} sessions")
    return r2_active, r2_spontaneous, r2_passive

def calculate_r2_attribution_correlation(attribution_data, r2_data):
    """Calculate correlation between R² and total attribution per channel/band.
    
    Args:
        attribution_data: Dictionary of attribution arrays
        r2_data: Dictionary of R² arrays
        
    Returns:
        tuple: (correlations, sessions)
    """
    print("Calculating R² - attribution correlations...")
    
    common_sessions = set(attribution_data.keys()) & set(r2_data.keys())
    sessions = sorted(list(common_sessions))
    
    correlations = np.zeros((len(sessions), len(config.BANDS) + 1))
    
    for i, session in enumerate(sessions):
        attr = attribution_data[session]
        r2 = r2_data[session]
        
        # Calculate total attribution per channel for each band
        total_attr = np.nansum(np.abs(attr), axis=2)  # sum over units
        
        for band_idx in range(len(config.BANDS) + 1):
            attr_vals = total_attr[:, band_idx]
            r2_vals = r2[:, band_idx]
            
            # Remove invalid values
            valid_mask = (
                (~np.isnan(attr_vals)) & (~np.isinf(attr_vals)) & 
                (~np.isnan(r2_vals)) & (~np.isinf(r2_vals))
            )
            
            if np.sum(valid_mask) > 1:
                corr, _ = stats.pearsonr(attr_vals[valid_mask], r2_vals[valid_mask])
                correlations[i, band_idx] = corr
            else:
                correlations[i, band_idx] = np.nan
    
    return correlations, sessions

In [ ]:
def apply_r2_bias_correction(attribution_data, r2_data):
    """Apply regression-based correction to remove R² bias from attribution scores.
    
    Args:
        attribution_data: Dictionary of attribution arrays
        r2_data: Dictionary of R² arrays
        
    Returns:
        dict: Corrected attribution data
    """
    print("Applying R² bias correction...")
    
    common_sessions = set(attribution_data.keys()) & set(r2_data.keys())
    corrected_attribution = {}
    
    for session in common_sessions:
        attr = attribution_data[session].copy()
        r2 = r2_data[session]
        
        n_channels, n_bands, n_units = attr.shape
        corrected_attr = np.zeros_like(attr)
        
        for band_idx in range(n_bands):
            # R² values for this band
            x = r2[:, band_idx]
            
            # Calculate total attribution per channel
            total_attr_per_channel = np.nansum(np.abs(attr[:, band_idx, :]), axis=1)
            
            # Valid data mask
            valid_mask = (
                (~np.isnan(total_attr_per_channel)) & (~np.isinf(total_attr_per_channel)) &
                (~np.isnan(x)) & (~np.isinf(x))
            )
            
            if np.sum(valid_mask) > 1 and np.ptp(x[valid_mask]) > 0:
                # Regression: total_attribution ~ R²
                slope, intercept, _, _, _ = stats.linregress(
                    x[valid_mask], total_attr_per_channel[valid_mask]
                )
                
                # Calculate residuals
                y_pred = intercept + slope * x
                residuals = total_attr_per_channel - y_pred
                corrected_total = residuals + np.nanmean(total_attr_per_channel[valid_mask])
                
                # Redistribute corrected totals to individual neurons proportionally
                for ch in range(n_channels):
                    if valid_mask[ch] and total_attr_per_channel[ch] > 0:
                        # Proportional redistribution preserving signs
                        neuron_props = np.abs(attr[ch, band_idx, :]) / total_attr_per_channel[ch]
                        
                        for unit_idx in range(n_units):
                            original_sign = np.sign(attr[ch, band_idx, unit_idx])
                            corrected_attr[ch, band_idx, unit_idx] = (
                                original_sign * neuron_props[unit_idx] * corrected_total[ch]
                            )
                    else:
                        corrected_attr[ch, band_idx, :] = attr[ch, band_idx, :]
            else:
                # No correction possible, keep original
                corrected_attr[:, band_idx, :] = attr[:, band_idx, :]
        
        corrected_attribution[session] = corrected_attr
    
    print(f"Applied bias correction to {len(corrected_attribution)} sessions")
    return corrected_attribution

In [ ]:
def plot_r2_scores_overview(r2_active, r2_spontaneous, r2_passive):
    """Plot comprehensive overview of R² scores across conditions.
    
    Args:
        r2_active: Active R² scores dictionary
        r2_spontaneous: Spontaneous R² scores dictionary  
        r2_passive: Passive R² scores dictionary
    """
    # Calculate average R² scores across sessions
    r2_avg_active = np.zeros((config.CHANNELS_TO_KEEP, len(config.BANDS) + 1))
    r2_avg_spont = np.zeros((config.CHANNELS_TO_KEEP, len(config.BANDS) + 1))
    r2_avg_passive = np.zeros((config.CHANNELS_TO_KEEP, len(config.BANDS) + 1))
    
    # Stack all sessions for averaging
    active_all = np.stack(list(r2_active.values()), axis=0)
    spont_all = np.stack(list(r2_spontaneous.values()), axis=0)
    passive_all = np.stack(list(r2_passive.values()), axis=0)
    
    r2_avg_active = np.nanmean(active_all, axis=0)
    r2_avg_spont = np.nanmean(spont_all, axis=0)
    r2_avg_passive = np.nanmean(passive_all, axis=0)
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Plot heatmaps
    conditions = [
        (r2_avg_active, 'Active', axes[0]),
        (r2_avg_passive, 'Passive', axes[1]),
        (r2_avg_spont, 'Spontaneous', axes[2])
    ]
    
    for r2_data, condition, ax in conditions:
        im = ax.imshow(r2_data, aspect='auto', cmap='viridis', vmin=0, vmax=0.7)
        ax.set_title(f'Average R² Scores - {condition}')
        ax.set_xlabel('Frequency Bands')
        ax.set_ylabel('Channels')
        
        # Set band labels
        ax.set_xticks(np.arange(len(config.BANDS) + 1))
        ax.set_xticklabels(['Broadband'] + [f'{band[0]}-{band[1]}' for band in config.BANDS], 
                          rotation=45, ha='right')
        ax.set_yticks(np.arange(config.CHANNELS_TO_KEEP))
        ax.set_yticklabels(np.arange(1, config.CHANNELS_TO_KEEP + 1))
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)
        cbar.set_label('R² Score', rotation=270, labelpad=15)
    
    plt.tight_layout()
    if config.FIGURES_PATH.exists():
        plt.savefig(config.FIGURES_PATH / 'r2_scores_overview.pdf', bbox_inches='tight')
    plt.show()
    
    # Plot broadband R² comparison across channels
    fig, ax = plt.subplots(figsize=(10, 6))
    
    channels = np.arange(1, config.CHANNELS_TO_KEEP + 1)
    
    # Calculate standard errors
    active_sem = stats.sem(active_all[:, :, 0], axis=0, nan_policy='omit')
    passive_sem = stats.sem(passive_all[:, :, 0], axis=0, nan_policy='omit')
    spont_sem = stats.sem(spont_all[:, :, 0], axis=0, nan_policy='omit')
    
    # Plot with error bars
    ax.plot(channels, r2_avg_active[:, 0], 'k-', linewidth=2, label='Active')
    ax.fill_between(channels, r2_avg_active[:, 0] - active_sem, 
                   r2_avg_active[:, 0] + active_sem, alpha=0.3, color='black')
    
    ax.plot(channels, r2_avg_passive[:, 0], 'b-', linewidth=2, label='Passive')
    ax.fill_between(channels, r2_avg_passive[:, 0] - passive_sem,
                   r2_avg_passive[:, 0] + passive_sem, alpha=0.3, color='blue')
    
    ax.plot(channels, r2_avg_spont[:, 0], 'r-', linewidth=2, label='Spontaneous')
    ax.fill_between(channels, r2_avg_spont[:, 0] - spont_sem,
                   r2_avg_spont[:, 0] + spont_sem, alpha=0.3, color='red')
    
    # Add layer boundaries
    ax.axvline(6.5, color='gray', linestyle='--', alpha=0.7)
    ax.axvline(10.5, color='gray', linestyle='--', alpha=0.7)
    
    ax.set_xlim(1, config.CHANNELS_TO_KEEP)
    ax.set_xlabel('Channel Number')
    ax.set_ylabel('R² Score')
    ax.set_title('Broadband R² Scores Across Channels')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    if config.FIGURES_PATH.exists():
        plt.savefig(config.FIGURES_PATH / 'broadband_r2_comparison.pdf', bbox_inches='tight')
    plt.show()

def plot_r2_correlation_analysis(correlations, sessions):
    """Plot R² vs attribution correlation analysis.
    
    Args:
        correlations: Correlation matrix (sessions x bands)
        sessions: List of session IDs
    """
    fig, ax = plt.subplots(1, 1, figsize=(16, 5))
    
    # Plot individual session correlations
    ax.plot(np.arange(len(config.BANDS) + 1), correlations.T, '.', 
            color='k', alpha=0.2, markersize=3)
    
    # Plot mean correlation
    mean_corr = np.nanmean(correlations, axis=0)
    sem_corr = stats.sem(correlations, axis=0, nan_policy='omit')
    
    ax.plot(np.arange(len(config.BANDS) + 1), mean_corr, 'b-', linewidth=2)
    ax.fill_between(np.arange(len(config.BANDS) + 1), 
                    mean_corr - sem_corr, mean_corr + sem_corr,
                    alpha=0.5, color='b')
    
    # Formatting
    ax.set_xticks(np.arange(len(config.BANDS) + 1))
    ax.set_xticklabels(['Broadband'] + [f'{band[0]}-{band[1]} Hz' for band in config.BANDS], 
                       rotation=45, ha='right')
    ax.grid(visible=True)
    ax.set_ylabel("Correlation Coefficient")
    ax.set_title("R² vs Attribution Correlation Across Sessions")
    ax.axhline(0, color='red', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    if config.FIGURES_PATH.exists():
        plt.savefig(config.FIGURES_PATH / 'r2_attr_correlation.pdf', bbox_inches='tight')
    plt.show()
    
    print(f"Mean correlation across bands: {np.nanmean(mean_corr):.3f}")
    print(f"Range: {np.nanmin(mean_corr):.3f} to {np.nanmax(mean_corr):.3f}")

## 8. R² Analysis Execution

In [ ]:
# Execute R² analysis pipeline
print("=== R² Analysis Pipeline ===")

# Load R² data
print("\n1. Loading R² scores...")
r2_active, r2_spontaneous, r2_passive = load_r2_scores(borders_aligned, visp_depths_df)

if r2_active:
    print(f"   ✓ Loaded R² data for {len(r2_active)} sessions")
    
    # Plot R² overview
    print("\n2. Creating R² overview plots...")
    plot_r2_scores_overview(r2_active, r2_spontaneous, r2_passive)
    
    # Calculate correlations with attribution
    print("\n3. Analyzing R² vs Attribution correlations...")
    correlations, session_list = calculate_r2_attribution_correlation(attribution_data, r2_active)
    plot_r2_correlation_analysis(correlations, session_list)
    
    # Apply bias correction
    print("\n4. Applying R² bias correction...")
    corrected_attribution = apply_r2_bias_correction(attribution_data, r2_active)
    corrected_spont_attribution = apply_r2_bias_correction(spont_attribution_data, r2_spontaneous)
    
    # Verify correction effectiveness
    print("\n5. Verifying correction effectiveness...")
    corrected_correlations, _ = calculate_r2_attribution_correlation(corrected_attribution, r2_active)
    
    print(f"   Original correlation range: {np.nanmin(correlations):.3f} to {np.nanmax(correlations):.3f}")
    print(f"   Corrected correlation range: {np.nanmin(corrected_correlations):.3f} to {np.nanmax(corrected_correlations):.3f}")
    print(f"   Mean correlation reduction: {np.nanmean(np.abs(correlations)) - np.nanmean(np.abs(corrected_correlations)):.3f}")
    
    # Plot corrected correlations
    plot_r2_correlation_analysis(corrected_correlations, session_list)
    
    print("\n✓ R² Analysis Complete!")
    print(f"   - {len(r2_active)} sessions analyzed")
    print(f"   - Bias correction applied successfully") 
    print(f"   - Corrected data ready for downstream analysis")
    
else:
    print("   ⚠ No R² data found. Check file paths and data availability.")
    corrected_attribution = attribution_data
    corrected_spont_attribution = spont_attribution_data

In [ ]:
def load_ablation_r2_data(borders_df, visp_depths_df):
    """Load ablation experiment R² results.
    
    Args:
        borders_df: DataFrame with border information
        visp_depths_df: DataFrame with channel depths
        
    Returns:
        tuple: (firing_rate_ablation, attribution_ablation, random_ablation, steps)
    """
    print("Loading ablation R² data...")
    
    steps = np.arange(5, 100, 5)  # Ablation percentages
    
    r2_firing_rate = []
    r2_attribution = []
    r2_random = []
    
    for _, row in borders_df.iterrows():
        session = row['session']
        
        if session not in visp_depths_df['session'].values:
            continue
            
        # Define ablation result files
        r2_fr_file = config.R2_SCORES_PATH / f'{session}_r2_scores_fr.npy'
        r2_attr_file = config.R2_SCORES_PATH / f'{session}_r2_scores_attr.npy'
        r2_rand_file = config.R2_SCORES_PATH / f'{session}_r2_scores_rand.npy'
        
        if all([f.exists() for f in [r2_fr_file, r2_attr_file, r2_rand_file]]):
            # Load ablation results
            fr_scores = np.load(r2_fr_file).squeeze().reshape(-1)
            attr_scores = np.load(r2_attr_file).squeeze().reshape(-1)
            rand_scores = np.mean(np.load(r2_rand_file), axis=1).squeeze().reshape(-1)
            
            r2_firing_rate.append(fr_scores)
            r2_attribution.append(attr_scores)
            r2_random.append(rand_scores)
    
    if r2_firing_rate:
        r2_firing_rate = np.vstack(r2_firing_rate)
        r2_attribution = np.vstack(r2_attribution)
        r2_random = np.vstack(r2_random)
        
        print(f"   Loaded ablation data for {len(r2_firing_rate)} sessions")
        return r2_firing_rate, r2_attribution, r2_random, steps
    else:
        print("   No ablation data found")
        return None, None, None, steps

def plot_ablation_analysis(r2_fr, r2_attr, r2_rand, steps):
    """Plot ablation experiment results.
    
    Args:
        r2_fr: Firing rate ablation results
        r2_attr: Attribution-based ablation results 
        r2_rand: Random ablation results
        steps: Ablation percentages
    """
    if r2_fr is None:
        print("No ablation data to plot")
        return
        
    # Calculate means and standard errors
    fr_mean = np.mean(r2_fr, axis=0)
    fr_sem = np.std(r2_fr, axis=0) / np.sqrt(r2_fr.shape[0])
    
    attr_mean = np.mean(r2_attr, axis=0)
    attr_sem = np.std(r2_attr, axis=0) / np.sqrt(r2_attr.shape[0])
    
    rand_mean = np.mean(r2_rand, axis=0)
    rand_sem = np.std(r2_rand, axis=0) / np.sqrt(r2_rand.shape[0])
    
    # Create plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot with error bands
    ax.plot(steps, fr_mean, 'k-', linewidth=2, label='Firing Rate Ablation')
    ax.fill_between(steps, fr_mean - fr_sem, fr_mean + fr_sem, 
                   color='gray', alpha=0.5)
    
    ax.plot(steps, attr_mean, 'b-', linewidth=2, label='Attribution-based Ablation')
    ax.fill_between(steps, attr_mean - attr_sem, attr_mean + attr_sem,
                   color='lightblue', alpha=0.5)
    
    ax.plot(steps, rand_mean, 'g-', linewidth=2, label='Random Ablation')
    ax.fill_between(steps, rand_mean - rand_sem, rand_mean + rand_sem,
                   color='lightgreen', alpha=0.5)
    
    ax.set_xlabel('Percentage of Neurons Ablated (%)')
    ax.set_ylabel('R² Score')
    ax.set_title('Model Performance Degradation Under Ablation')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_ylim(bottom=0)
    
    plt.tight_layout()
    if config.FIGURES_PATH.exists():
        plt.savefig(config.FIGURES_PATH / 'ablation_analysis.pdf', bbox_inches='tight')
    plt.show()
    
    # Print summary statistics
    print(f"Performance at 50% ablation:")
    print(f"  Firing Rate: {fr_mean[9]:.3f} ± {fr_sem[9]:.3f}")  # 50% is step 9
    print(f"  Attribution: {attr_mean[9]:.3f} ± {attr_sem[9]:.3f}")
    print(f"  Random: {rand_mean[9]:.3f} ± {rand_sem[9]:.3f}")

# Execute ablation analysis if data available
print("\n=== Ablation Analysis ===")
r2_fr, r2_attr, r2_rand, ablation_steps = load_ablation_r2_data(borders_aligned, visp_depths_df)
plot_ablation_analysis(r2_fr, r2_attr, r2_rand, ablation_steps)

## 9. R² Analysis Summary

The R² analysis module provides comprehensive evaluation of model performance:

### ✅ Key Features Implemented:

1. **Multi-condition R² Loading**: Loads and aligns R² scores for active, spontaneous, and passive conditions
2. **Correlation Analysis**: Quantifies the relationship between R² scores and attribution magnitudes  
3. **Bias Correction**: Applies regression-based correction to remove R²-attribution correlations
4. **Comprehensive Visualization**: Creates heatmaps and line plots showing R² patterns across channels and frequency bands
5. **Ablation Analysis**: Evaluates model performance degradation under systematic neuron removal

### 📊 Generated Plots:
- **R² Overview Heatmaps**: Shows performance across channels and frequency bands for each condition
- **Broadband Comparison**: Line plots comparing R² across cortical depth for different conditions  
- **Correlation Analysis**: Demonstrates R²-attribution correlations before and after correction
- **Ablation Results**: Shows how different ablation strategies affect model performance

### 🎯 Analysis Outcomes:
- Identifies which channels and frequency bands are best predicted by the model
- Reveals layer-specific patterns in model performance
- Validates the effectiveness of bias correction procedures
- Quantifies the relative importance of different neuron subpopulations

The corrected attribution data is now ready for downstream analysis without R² confounds.